In [25]:
import math
import random

In [26]:
s0 = 100
annual_Volatility = 0.3
r = 0.05
time = 9/12
X1 = 120
X2 = 76


In [27]:
def bs():
    X1_call = 0
    X1_put = 0
    X2_call = 0
    X2_put = 0

    N = lambda x: 0.5 * (1 + math.erf(x / math.sqrt(2)))

    d1_X1 = (math.log(s0 / X1) + (r + (annual_Volatility ** 2) / 2) * time) / (annual_Volatility * math.sqrt(time))
    d2_X1 = d1_X1 - annual_Volatility * math.sqrt(time)

    d1_X2 = (math.log(s0 / X2) + (r + (annual_Volatility ** 2) / 2) * time) / (annual_Volatility * math.sqrt(time))
    d2_X2 = d1_X2 - annual_Volatility * math.sqrt(time)

    X1_call = s0 * N(d1_X1) - X1 * math.exp(-r * time) * N(d2_X1)
    X1_put = X1 * math.exp(-r * time) * N(-d2_X1) - s0 * N(-d1_X1)

    X2_call = s0 * N(d1_X2) - X2 * math.exp(-r * time) * N(d2_X2)
    X2_put = X2 * math.exp(-r * time) * N(-d2_X2) - s0 * N(-d1_X2)

    print("Call value of X1: ", X1_call)
    print("Put value of X1: ", X1_put)
    print("Call value of X2: ", X2_call)
    print("Put value of X2: ", X2_put)


In [28]:
def trees(S0=100, sigma=0.30, r=0.05, T=0.75, X1=120, X2=76): 
    
    # 1. Enforce the ~1 day time step rule 
    steps = 274  # 0.75 years * 365 days = 273.75
    dt = T / steps
    
    # 2. Calculate tree parameters
    u = math.exp(sigma * math.sqrt(dt))
    d = 1 / u
    p = (math.exp(r * dt) - d) / (u - d)
    discount = math.exp(-r * dt)

    def binomial_price(X, is_call, is_american):
        option_values = []

        # Initialize final nodes at maturity
        for i in range(steps + 1):
            stock_price = S0 * (u ** (steps - i)) * (d ** i)
            if is_call:
                payoff = max(stock_price - X, 0)
            else:
                payoff = max(X - stock_price, 0)
            option_values.append(payoff)

        # Backward induction
        for step in range(steps - 1, -1, -1):
            new_values = []
            for i in range(step + 1):
                continuation = discount * (p * option_values[i] + (1 - p) * option_values[i + 1])

                if is_american:
                    stock_price = S0 * (u ** (step - i)) * (d ** i)
                    if is_call:
                        exercise = max(stock_price - X, 0)
                    else:
                        exercise = max(X - stock_price, 0)
                    new_values.append(max(continuation, exercise))
                else:
                    new_values.append(continuation)

            option_values = new_values

        return option_values

    # Calculating the 8 required options
    X1_euro_call = binomial_price(X1, True, False)
    X1_euro_put = binomial_price(X1, False, False)
    X1_american_call = binomial_price(X1, True, True)
    X1_american_put = binomial_price(X1, False, True)

    X2_euro_call = binomial_price(X2, True, False)
    X2_euro_put = binomial_price(X2, False, False)
    X2_american_call = binomial_price(X2, True, True)
    X2_american_put = binomial_price(X2, False, True)

    print("Call value of European X1: ", X1_euro_call)
    print("Put value of European X1: ", X1_euro_put)
    print("Call value of American X1: ", X1_american_call)
    print("Put value of American X1: ", X1_american_put)

    print("Call value of European X2: ", X2_euro_call)
    print("Put value of European X2: ", X2_euro_put)
    print("Call value of American X2: ", X2_american_call)
    print("Put value of American X2: ", X2_american_put)

    # Return the values so an automated grader can read them
    return (X1_euro_call, X1_euro_put, X1_american_call, X1_american_put,
            X2_euro_call, X2_euro_put, X2_american_call, X2_american_put)


In [29]:
def montecarlo(S0=100, sigma=0.30, r=0.05, T=0.75, X1=120, X2=76, num_simulations=10000, seed=1):
    steps = round(T * 365)
    dt = T / steps
    random.seed(seed)

    X1_call_payoffs = []
    X1_put_payoffs = []
    X2_call_payoffs = []
    X2_put_payoffs = []

    for _ in range(num_simulations):
        stock_price = S0

        for _ in range(steps):
            z = random.gauss(0, 1)
            stock_price = stock_price * math.exp((r - 0.5 * sigma ** 2) * dt + sigma * math.sqrt(dt) * z)

        X1_call_payoffs.append(max(stock_price - X1, 0))
        X1_put_payoffs.append(max(X1 - stock_price, 0))
        X2_call_payoffs.append(max(stock_price - X2, 0))
        X2_put_payoffs.append(max(X2 - stock_price, 0))

    discount = math.exp(-r * T)

    X1_euro_call = discount * (sum(X1_call_payoffs) / num_simulations)
    X1_euro_put = discount * (sum(X1_put_payoffs) / num_simulations)
    X2_euro_call = discount * (sum(X2_call_payoffs) / num_simulations)
    X2_euro_put = discount * (sum(X2_put_payoffs) / num_simulations)

    print("Call value of X1: ", X1_euro_call)
    print("Put value of X1: ", X1_euro_put)
    print("Call value of X2: ", X2_euro_call)
    print("Put value of X2: ", X2_euro_put)

    return (X1_euro_call, X1_euro_put, X2_euro_call, X2_euro_put)

In [30]:
bs()

Call value of X1:  5.023680857647037
Put value of X1:  20.607010984145646
Call value of X2:  28.035603930215792
Put value of X2:  1.2383796769982478


In [31]:
trees()

Call value of European X1:  [5.0236790423847815]
Put value of European X1:  [20.607009168883607]
Call value of American X1:  [5.0236790423847815]
Put value of American X1:  [21.927198754046557]
Call value of European X2:  [28.036653570316254]
Put value of European X2:  [1.2394293170985606]
Call value of American X2:  [28.036653570316254]
Put value of American X2:  [1.2723279060255912]


([5.0236790423847815],
 [20.607009168883607],
 [5.0236790423847815],
 [21.927198754046557],
 [28.036653570316254],
 [1.2394293170985606],
 [28.036653570316254],
 [1.2723279060255912])

In [32]:
montecarlo()

Call value of X1:  4.940544565565921
Put value of X1:  20.556602233207638
Call value of X2:  28.028963111771606
Put value of X2:  1.26446639969716


(4.940544565565921, 20.556602233207638, 28.028963111771606, 1.26446639969716)